## Create seasonal, multiannual SBAS networks from groups of Sentinel-1 burst data

This notebook focuses on using the `SBASNetwork` class to create SBAS networks of interferometric pairs from groups of adjacent Sentinel-1 burst data. Refer to the [SBASNetwork Tutorial](SBASNetwork.ipynb) for examples of creating SBASNetworks of networks of full Sentinel-1 scene or individual Sentinel-1 burst locations.

In [ ]:
# This is useful if you are working with a dev install of asf_search and experimenting with changes to the codebase
%load_ext autoreload
%autoreload 2

### Create an asf_search.S1MultiBurstGroup object defining a group of Sentinel-1 bursts

`asf_search.S1MultiBurstGroup` objects contains the member variable `bursts` dictionary, which maps burst IDs to a list of swaths included in the group.

A "full burst ID" is composed of 3 underscore-separated parts:

`ppp_dddddd_sss`

- `p`: Relative orbit/path
- `d`: Relative burst ID
- `s`: Sub-swath

Dictionary keys are comprised of the relative orbit and realtive burst IDs for each burst in the group: `ppp_dddddd`. 

Dictionary values are tuples of sub-swaths to include for each burst. Valid sub-swaths are "IW1", "IW2", and "IW3".

Refer to [HyP3's mutli-burst guidelines](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#considerations-for-selecting-input-bursts) when assembling a collection of bursts.

In [ ]:
import asf_search as asf

multiburst_group = asf.S1MultiBurstGroup(
    bursts={
    "173_370305": ("IW1", "IW2", "IW3"),
    "173_370306": ("IW1", "IW2", "IW3"),
    "173_370307": ("IW1", "IW2", "IW3")
    }
)

### Define the temporal and SBAS paramaters for the network


In [ ]:
from datetime import datetime
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = get_julian_season(("1-1", "6-25"))
start_date = '2023-01-01'
end_date = '2025-10-02'    
perpendicular_baseline=100
inseason_temporal_baseline=24
bridge_target_date='3-1'
bridge_year_threshold=2

### Create an SBASNetwork from the multi-burst group using the `SBASNetwork.s1_multiburst` class method

Multi-burst group validation is performed using the [burst2safe package](https://github.com/ASFHyP3/burst2safe). This identifies invalid burst collections prior to ordering them from HyP3.

When a multi-burst SBASNetwork is generated, matching SBASNetworks are formed for each burst in the multi-burst collection. Only interferometric burst pairs that appear in all burst SBASNetworks may be included. A burst pair may appear in one burst network and not another due to differences in baseline values or to a lack of coverage in the archive. If burst pairs are missing from any of the associated networks, the pairs will be removed from all networks.  




In [ ]:
%%time

multiburst_sbas = asf.S1MultiBurstSBASNetwork(
    multiburst_group, 
    start_date = start_date,
    end_date = end_date,
    season = season,
    perpendicular_baseline=perpendicular_baseline, 
    inseason_temporal_baseline=inseason_temporal_baseline,
    bridge_target_date=bridge_target_date,
    bridge_year_threshold=bridge_year_threshold
)


multiburst_sbas

### The multi-burst SBASNetwork contains a list of corresponding single-burst SBASNetworks that comprise the multi-burst collection: `sbas.s1_multiburst_sbas_networks`

In [ ]:
print(f"multiburst_sbas contains {len(multiburst_sbas.sbas_networks)} sbas_networks")

### The multi-burst SBASNetwork also contains a list of geographic reference scenes for each of the single-burst networks

In [ ]:
multiburst_sbas.georeferences

### Plot one of the multi-burst SBASNetwork's single-burst SBASNetworks

Every single-burst network in a multi-burst network contains corresponding date pairs, and their plots will be identical.

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

### Plot another single-burst `SBASNetwork`

Notice that both network plots appear identical and contain corresponding InSAR pairs.

In [ ]:
multiburst_sbas.sbas_networks[1].plot()

<hr>

## Add custom Pairs to the network.

### Create a S1MultiBurstSBASPairMap object

S1MultiBurstSBASPairMap objects contain a geo_reference dictionary member variable, which maps each `S1MultiBurstSBASPairMap.sbas_network`'s `geo_reference` products to Pairs that will be added or removed from their `subset_stack`s

When creating a dictionary of pairs to add. We associate each pair list to its single-burst stack using its geographic reference product:
```
{
    geo_reference_burst_1: [Pair_1, Pair_2],
    geo_reference_burst_2: [Pair_3, Pair_4],
    geo_reference_burst_3: [Pair_5, Pair_6],
}
```


Below, we select Pairs from the remove lists by date using the `get_pair_from_dates` method. In this example, we create the geo_reference dictionary by finding pairs with the desired dates in each single-burst SBASNetwork's remove_list.

In [ ]:
try:
    from ciso8601 import parse_datetime
except ImportError:
    from dateutil.parser import parse as parse_datetime

# Creates the geo_reference dictionary by finding pairs with the desired dates in each single-burst SBASNetwork's remove_list
geo_reference_dict_1 = {s.geo_reference: [
    asf.get_pair_from_dates(s.remove_list, parse_datetime("20230304").date(), parse_datetime("20240614").date()),
    asf.get_pair_from_dates(s.remove_list, parse_datetime("20230127").date(), parse_datetime("20230304").date()),
    ] for s in multiburst_sbas.sbas_networks}

multiburst_geo_ref_pair_map_1 = asf.S1MultiBurstSBASPairMap(geo_reference_dict_1, multiburst_sbas.sbas_networks)
multiburst_geo_ref_pair_map_1

### Add the identified pairs back to the SBASNetwork

In [ ]:
multiburst_sbas.add_pairs(multiburst_geo_ref_pair_map_1)

### Plot the SBASNetwork

Note that the previously disconnected networks are now connected with a new pair (2023/03/04 - 2024/06/14). 

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

### Add Pairs that are completly unknown to the SBASNetwork

Search for Pairs to add that were outside the bounds of the original SBASNetwork's time range and therefor are not on any remove_lists. 

In [ ]:
geo_reference_dict_2 = dict()
for geo_ref in multiburst_sbas.georeferences:
    results = geo_ref.stack()
    burst_slcs = []
    for result in results:
        if parse_datetime(result.properties["startTime"]).date() == parse_datetime("2026-05-17").date() or \
            parse_datetime(result.properties["startTime"]).date() == parse_datetime("2026-05-23").date():
            burst_slcs.append(result)
    geo_reference_dict_2[geo_ref] = [asf.Pair(geo_ref, burst_slcs[0]), asf.Pair(geo_ref, burst_slcs[1]), asf.Pair(burst_slcs[0], burst_slcs[1])]

multiburst_geo_ref_pair_map_2 = asf.S1MultiBurstSBASPairMap(geo_reference_dict_2, multiburst_sbas.sbas_networks)
multiburst_geo_ref_pair_map_2

multiburst_sbas.add_pairs(multiburst_geo_ref_pair_map_2)

### Plot the SBASNetwork again after adding more pairs

Notice that the SBASNetwork has expanded to include custom date pairs in 2026.

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

<hr>

## Remove date pairs from the SBASNetwork

### Remove the first set of pairs we added

In [ ]:
multiburst_sbas.remove_pairs(multiburst_geo_ref_pair_map_2)

### Plot the SBASNetwork, once again broken into two disconnected networks

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

<hr>

## Use the `get_scene_ids` method to access multi-burst product IDs for easy InSAR product ordering from HyP3 

In [ ]:
sbas_burst_ids = multiburst_sbas.get_scene_ids()
sbas_burst_ids